significant trials, significant GT \n
plot trials based on significance or sleep state

In [1]:
import os
import numpy as np
import mne
import imageio
import h5py
# import scipy.fftpack
import matplotlib
import pywt
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
# from scipy import signal
from matplotlib.colors import ListedColormap
import time
import seaborn as sns

# import scipy.io as sio
# from scipy.integrate import simps
import pandas as pd
# from scipy import fft
import matplotlib.mlab as mlab
import sys
import matplotlib as mpl
sys.path.append('T:\EL_experiment\Codes\CCEP_human\Python_Analysis\py_functions')
import NMF_funcs as NMFf
import significant_connections as SCF
from scipy.stats import norm
import LL_funcs as LLf
from scipy.stats import norm
from tkinter import filedialog
from tkinter import *
import ntpath
import supp_sleep_CCEP_figures as fig_sleep
root = Tk()
root.withdraw()
import math
import scipy
from scipy import signal
import pylab
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
import platform
from glob import glob
from scipy.io import savemat
import scipy.cluster.hierarchy as spc
from scipy.spatial import distance
from sklearn.cluster import KMeans
import h5py
import basic_func as bf
from scipy.integrate import simps
from numpy import trapz
#import IO_func as IOF
#import BM_func as BMf
import tqdm
from matplotlib.patches import Rectangle
from pathlib import Path
sub_path  ='X:\\4 e-Lab\\' # y:\\eLab
import BM_plots as BMp
import freq_funcs as ff
import CCEP_plot
import supp_CCEP_figures as CCEP_supp
import random
import math
from sklearn.metrics import cohen_kappa_score, jaccard_score, accuracy_score
dist_groups = np.array([[0, 30], [30, 60], [60, 120]])
dist_labels = ['local (<30 mm)', 'short (<60mm)', 'long']
Fs = 500
dur = np.zeros((1, 2), dtype=np.int32)
t0 = 1
dur[0, 0] = -t0
dur[0, 1] = 3

folder = 'BrainMapping'
# dur[0,:]       = np.int32(np.sum(abs(dur)))
x_ax = np.arange(dur[0, 0], dur[0, 1], (1 / Fs))
color_elab = np.zeros((3, 3))
color_elab[0, :] = np.array([31, 78, 121]) / 255
color_elab[1, :] = np.array([189, 215, 238]) / 255
color_elab[2, :] = np.array([0.256, 0.574, 0.431])
cwd = os.getcwd()
color_sleep = ['#808080', '#145da0', '#ff1919']
label_sleep = ['Wake', 'NREM', 'REM']
color_dist = ['0000FF','#0076C4','#00DD91']
import CCEP_func

In [2]:
from scipy.signal import savgol_filter, find_peaks, peak_prominences

In [3]:
cond_folder = 'CR'
path_gen_base = sub_path + '\Patients'

In [4]:
CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
tab_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='plot')
tab_region = tab_region.sort_values('Order').reset_index(drop=True)
regions_all = tab_region.Area.values
region_col = tab_region.color.values

CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
all_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='atlas')

In [5]:
plt.rcParams.update({
            'font.family': 'arial',
            'font.size': 12,
            'xtick.labelsize': 8,
            'ytick.labelsize': 8,
            'legend.fontsize': 9,
            'svg.fonttype': 'none',
            'font.size': 10,
            'axes.titlesize': 10,
            'axes.labelsize': 8,
            'xtick.labelsize': 8,
            'ytick.labelsize': 8,
            'legend.fontsize': 9,
            'figure.titlesize': 10
        })

In [6]:
subj = "EL011"
path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)

path_gen = os.path.join(sub_path+'\Patients\\' + subj)
if not os.path.exists(path_gen):
    path_gen = 'T:\\EL_experiment\\Patients\\' + subj
path_patient = path_gen + '\Data\EL_experiment'  # os.path.dirname(os.path.dirname(cwd))+'/Patients/'+subj
path_infos = os.path.join(path_gen, 'Electrodes')
# labels
files_list = glob(path_patient_analysis + '\\' + folder + '/data/Stim_list_*')
i = 0
stimlist_file = path_patient_analysis + '\\' + folder + '\\' + cond_folder + '\\data\\stimlist_' + cond_folder + '.csv'
stimlist = pd.read_csv(stimlist_file)
lbls = pd.read_excel(os.path.join(path_infos, subj + "_labels.xlsx"), header=0, sheet_name='BP')
if "type" in lbls.columns:
    lbls = lbls[lbls.type=='SEEG']
    lbls = lbls.reset_index(drop=True)
labels_all, labels_region, labels_clinic, coord_all, StimChans, StimChanSM, StimChansC, StimChanIx, stimlist = bf.get_Stim_chans(
    stimlist,
    lbls)
stimlist_sleep = pd.read_csv(os.path.join(path_patient_analysis, 'stimlist_hypnogram.csv'))
file_con = path_patient_analysis + '\\' + folder + '/' + cond_folder + '/data/con_trial_all.csv'
con_trial = pd.read_csv(file_con)
badchans = pd.read_csv(path_patient_analysis + '/BrainMapping/data/badchan.csv')
bad_chans = np.unique(np.array(np.where(badchans.values[:, 1:] == 1))[0, :])
con_trial = bf.add_sleepstate(con_trial)

In [7]:
file_CC_summ = path_patient_analysis + '\\' + folder + '\\data\\CC_summ_' + 'similarity' + '.csv'
CC_summ = pd.read_csv(file_CC_summ)


In [8]:
h5_file = path_patient_analysis + '\\' + folder + '\\' + cond_folder + '\\data\\EEG_' + cond_folder + '.h5'
if os.path.isfile(h5_file):
    print('loading h5')
    EEG_resp = h5py.File(h5_file)
    EEG_resp = EEG_resp['EEG_resp']

loading h5


In [9]:
h5_file = path_patient_analysis + '\\' + folder + '\\data\\M_CC_similarity.h5'
if os.path.isfile(h5_file):
    print('loading h5')
    CC = h5py.File(h5_file)
    CC = CC['M_GT_all']

loading h5


In [10]:
h5_file = path_patient_analysis + '\\' + folder + '\\data\\LL_CC_surr_similarity.h5'
if os.path.isfile(h5_file):
    print('loading h5')
    CC_surr = h5py.File(h5_file)
    CC_WOI = CC_surr['CC_WOI']
    CC_surr = CC_surr['CC_LL_surr']
    
h5_file = path_patient_analysis + '\\' + folder + '\\data\\M_CC_similarity.h5'
if os.path.isfile(h5_file):
    print('loading h5')
    CC = h5py.File(h5_file)
    CC = CC['M_GT_all']

loading h5
loading h5


In [11]:
file_CC_summ = path_patient_analysis + '\\' + folder + '\\' + cond_folder + '\\data\\summ_general.csv'  # summary_genera
con_summary_all = pd.read_csv(file_CC_summ)
con_summary_all = con_summary_all.drop_duplicates()

## Functions

In [15]:
def get_peak(signal):
    """
       Find the location of the first peak among the two strongest peaks in both polarities.

       Parameters:
       - signal (array): Time signal.

       Returns:
       - float: Location of the first peak in seconds.
       """
    # Find positive peaks and their prominences
    positive_peaks, _ = find_peaks(signal)
    positive_prominences = peak_prominences(signal, positive_peaks)[0]

    # Find negative peaks and their prominences
    negative_peaks, _ = find_peaks(-signal)
    negative_prominences = peak_prominences(-signal, negative_peaks)[0]

    # Handle the case when no peaks are found
    if len(positive_prominences) == 0:
        sorted_positive_peaks = np.array([np.nan])
    else:
        # Sort positive peaks by prominence in descending order
        sorted_positive_peaks = positive_peaks[np.argsort(-positive_prominences)]

    if len(negative_prominences) == 0:
        sorted_negative_peaks = np.array([np.nan])
    else:
        # Sort negative peaks by prominence in descending order
        sorted_negative_peaks = negative_peaks[np.argsort(-negative_prominences)]

    # Get the two strongest peaks from both polarities
    first_peak = np.nanmin([sorted_positive_peaks[0], sorted_negative_peaks[0]])

    return first_peak
def peak_latency(trials, WOI, t0=1, Fs=500, w_LL=0.25):
    BL_period = [int((t0 - 0.5) * Fs), int((t0 - 0.00) * Fs)]
    bl_median = np.median(trials[:, BL_period[0]:BL_period[1]], axis=1)
    trials = ff.lp_filter(trials, 45, Fs)
    trials = trials - bl_median[:, None]

    # 2. Average signal
    mean_signal = np.mean(trials, axis=0)

    # 3. Subtract BL median
    mean_signal = mean_signal - np.median(mean_signal[BL_period[0]:BL_period[1]])

    # 4. Calculate standard deviation of baseline period
    std = np.std(mean_signal[BL_period[0]:BL_period[1]])
    # 5. Threshold: +/- 3.4 std, find first peak crossing this threshold
    factor = 3.4
    mean_signal[:int((t0 + 0.015) * Fs)] = 0
    mean_signal[int((t0 + WOI + 2 * w_LL / 3) * Fs):] = 0

    threshold = factor * std
    peaks, _ = find_peaks(mean_signal, height=threshold)
    neg_peaks, _ = find_peaks(-mean_signal, height=threshold)
    peak_detected = 1
    # Finding the first peak crossing statistical threshold
    if peaks.size > 0 or neg_peaks.size > 0:
        all_peaks = np.sort(np.concatenate((peaks, neg_peaks)))
        first_peak = all_peaks[0]
    else:
        threshold = 2.5 * std
        peaks, _ = find_peaks(mean_signal, height=threshold)
        neg_peaks, _ = find_peaks(-mean_signal, height=threshold)
        # Finding the first peak crossing statistical threshold
        if peaks.size > 0 or neg_peaks.size > 0:
            all_peaks = np.sort(np.concatenate((peaks, neg_peaks)))
            first_peak = all_peaks[0]
        else:
            mean_signal = np.mean(trials, axis=0)
            mean_signal = mean_signal - np.median(mean_signal[BL_period[0]:BL_period[1]])
            mean_signal[:int((t0 + 0.010) * Fs)] = 0
            mean_signal[int((t0 + WOI + 0.3) * Fs):] = 0
            first_peak = get_peak(mean_signal)
            peak_detected = 0
    if np.isnan(first_peak):
        polarity = np.nan
    else:
        polarity = np.sign(mean_signal[int(first_peak)])
    return first_peak / Fs - t0, polarity, peak_detected


def peak_latency_finetuning(trials, WOI, peak_lat_general, polarity, t0=1, Fs=500, w_LL=0.25):
    BL_period = [int((t0 - 0.5) * Fs), int((t0 - 0.00) * Fs)]
    bl_median = np.median(trials[:, BL_period[0]:BL_period[1]], axis=1)
    trials = ff.lp_filter(trials, 45, Fs)
    trials = trials - bl_median[:, None]

    # 2. Average signal
    mean_signal = np.mean(trials, axis=0)

    # 3. Subtract BL median
    mean_signal = mean_signal - np.median(mean_signal[BL_period[0]:BL_period[1]])

    # 4. Calculate standard deviation of baseline period
    std = np.std(mean_signal[BL_period[0]:BL_period[1]])
    # 5. Threshold: +/- 3.4 std, find first peak crossing this threshold
    get_peak_check = 1
    factor = 1
    mean_signal[:int((t0 + 0.015) * Fs)] = 0
    mean_signal[int((t0 + WOI + 2 * w_LL / 3) * Fs)] = 0
    first_peak = None

    threshold = factor * std
    if polarity == 1:
        peaks, _ = find_peaks(mean_signal, prominence=threshold)
    else:
        peaks, _ = find_peaks(-mean_signal, prominence=threshold)

    if peaks.size > 0:
        # find closest peak to peak_lat_general
        peak_lat_general_datapoint = (t0 + peak_lat_general) * Fs
        first_peak = peaks[np.argmin(np.abs(peaks - peak_lat_general_datapoint))]
        peak_detected = 1
    else:
        if polarity == 1:
            peaks, _ = find_peaks(mean_signal, prominence=threshold/10)
        else:
            peaks, _ = find_peaks(-mean_signal, prominence=threshold/10)
        if peaks.size > 0:
            # find closest peak to peak_lat_general
            peak_lat_general_datapoint = (t0 + peak_lat_general) * Fs
            first_peak = peaks[np.argmin(np.abs(peaks - peak_lat_general_datapoint))]
            peak_detected = 1
        else:
            mean_signal[:int((t0 + peak_lat_general - 0.03) * Fs)] = 0
            mean_signal[int((t0 + peak_lat_general + 0.03) * Fs):] = 0
            np.argmax(polarity*mean_signal)
            peak_detected = 0

    return first_peak / Fs - t0, peak_detected


def CCEP_onset_finetuning(trials, onset_general, peak_lat_general, polarity, WOI=0, t0=1, Fs=500, w_LL=0.25):
    """
    Calculate the onset of a Cortico-Cortical Evoked Potential (CCEP) in a signal based on the second derivative.

    Parameters:
    - trials (array): all trials for given connection .
    - onset_general (float): Expected CCEP onset in seconds after stimulation (t0).
    - WOI (float): Onset of Window Of Interest based on previous LL calculations (connection-specific).
    - t0 (float): Time of stimulation in the signal (e.g., for epoch: [-1, 3] -> t0 = 1).
    - Fs (int): Sampling frequency.
    - w_LL (float): Window length for onset detection.

    Returns:
    - float: Time of response onset after stimulation, in seconds.
    - float: Peak latency in seconds.
    """
    signal = np.mean(trials, axis=0)

    # Calculate peak latency based on trials
    peak_lat, peak_det = peak_latency_finetuning(trials, WOI, peak_lat_general, polarity, t0=1, Fs=500, w_LL=0.25)

    t_onset, signal_derivative = get_onset_der(signal, peak_lat, polarity, t0=t0, Fs=Fs)

    return t_onset, peak_lat, peak_det


def get_onset_der(signal, peak_lat, polarity, t0 = 1, Fs = 500):
    kernel_size = int(0.1 * Fs) + 1
    signal_derivative = savgol_filter(signal, window_length=kernel_size, polyorder=3, deriv=2)
    #
    signal_derivative = signal_derivative - np.median(signal_derivative[int(0.5 * Fs):int(Fs * (t0 - 0.01))])
    BL_data = np.abs(signal_derivative)[int(0.5 * Fs):int(Fs * (t0 - 0.01))]
    factor = 2
    thr = factor * np.std(BL_data)
    if polarity == 1:
        peaks, _ = find_peaks(signal_derivative, height=thr)
    elif polarity == -1:
        peaks, _ = find_peaks(-signal_derivative, height=thr)
    else:
        peaks, _ = find_peaks(np.abs(signal_derivative), height=thr)
    # peaks = peaks[(peaks > int((t0 - 0.005) * Fs)) & (peaks < int(((t0 + peak_CCEP) * Fs)))]
    peaks = peaks[(peaks > int((t0 - 0.01) * Fs)) & (peaks < int(((t0 + peak_lat - 0.01) * Fs)))]
    peak_times = peaks / Fs - t0  # Adjusting for epoched time

    if len(peak_times) > 0:  # take strongest peak poststim (before N1 peak)
        t_onset = peak_times[np.argmin(peak_lat - peak_times)]  # peak_times[0]
        # t_onset =  peak_times[np.argmax(abs(signal_derivative[peaks]))]# peak_times[0]
    else:  # if there is no peak passing the threshold, take max within specifc window
        if polarity == 1:
            peaks, _ = find_peaks(signal_derivative)
        elif polarity == -1:
            peaks, _ = find_peaks(-signal_derivative)
        else:
            peaks, _ = find_peaks(np.abs(signal_derivative))
        # peaks = peaks[(peaks > int((t0 - 0.005) * Fs)) & (peaks < int(((t0 + peak_CCEP) * Fs)))]
        peaks = peaks[(peaks > int((t0 - 0.01) * Fs)) & (peaks < int(((t0 + peak_lat - 0.01) * Fs)))]
        peak_times = peaks / Fs - t0
        if len(peak_times) > 0:  # take strongest peak poststim (before N1 peak)
            t_onset = peak_times[np.argmin(peak_lat - peak_times)]  # peak_times[0]
        elif peak_lat < 0.05:
            t_onset = 0
        else:
            if polarity == 1:
                peaks = np.argmax(signal_derivative)
            elif polarity == -1:
                peaks = np.argmin(signal_derivative)
            else:
                peaks = np.argmax(abs(signal_derivative))

            t_onset = peaks /Fs - t0


    if t_onset < 0:
        t_onset = 0
    return t_onset, signal_derivative
def CCEP_onset(trials, WOI=0, t0=1, Fs=500, w_LL=0.25, plot=False):
    """
    Calculate the onset of a Cortico-Cortical Evoked Potential (CCEP) in a signal.

    Parameters:
    - trials (array): all trials for given connection .
    - WOI (float): Onset of Window Of Interest based on previous LL calculations (connection-specific).
    - t_0 (float): Time of stimulation in the signal (e.g., for epoch: [-1, 3] -> t_0 = 1).
    - Fs (int): Sampling frequency.
    - w_LL_onset (float): Window length for onset detection.

    Returns:
    - float: Time of response onset after stimulation, in seconds.
    """
    signal = np.mean(trials, 0)
    peak_lat, polarity, peak_detected = peak_latency(trials, WOI, t0=1, Fs=500, w_LL=0.25)
    t_onset, signal_derivative = get_onset_der(signal, peak_lat, polarity, t0=t0, Fs=Fs)

    if plot:
        x_ax = np.linspace(-1, 3, len(signal))
        plt.plot(x_ax, signal, color=[0, 0, 0])
        plt.plot(x_ax, signal_derivative * 100, color=[0, 0, 1])
        # plt.plot(x_ax + w_LL / 2, LL_transform * 100, color=[0, 0, 0], alpha=0.7)
        plt.axvline(0, color=[1, 0, 0])
        plt.axvline(peak_lat, color=[0, 0, 0])
        plt.axvline(t_onset, color=[0, 0, 0], ls='--')
        plt.xlim([-0.2, 0.7])
        plt.show()
    return t_onset, peak_lat, polarity, peak_detected




## Test

In [13]:
stop

NameError: name 'stop' is not defined

In [12]:
w_LL = 0.25

In [13]:
path  ='X:\\4 e-Lab\\EvM\Projects\EL_experiment\Analysis\Patients\Across\CCEP\peak_latency\\example_general'

In [17]:
t_onset, peak_lat, polarity, peak_detected = CCEP_func.CCEP_onset(EEG_resp[rc, stimnum, :], WOI=WOI, t0=1, Fs=500, w_LL=0.25, plot=False, skip_nonpeak = 0)

ValueError: too many values to unpack (expected 2)

In [16]:
sc = 38
rc = 49
lists = con_trial[(con_trial['Chan'] == rc) & (con_trial['Stim'] == sc)& (con_trial['Sig'] == 1)& (con_trial['Artefact'] <1)].reset_index(drop=True)
if len(lists)>10:
    stimnum = lists['Num'].values.astype('int')
    WOI = con_summary_all.loc[(con_summary_all.Stim == sc)&(con_summary_all.Chan ==rc), 't_WOI'].values[0]
    onset, peak, pol, _ = CCEP_onset(EEG_resp[rc, stimnum, :], WOI=0, t0=1, Fs=500, w_LL=0.25, plot=False)
    #onset = con_summary_all.loc[(con_summary_all.Stim == sc)&(con_summary_all.Chan ==rc), 'delay'].values[0]
    #peak = con_summary_all.loc[(con_summary_all.Stim == sc)&(con_summary_all.Chan ==rc), 'peak_latency'].values[0]
    fig, ax = plot_signals(EEG_resp[rc, stimnum, :], onset, peak, Fs=500, w_LL =0.25, t0 =1)
    plt.suptitle(labels_all[sc]+' - '+labels_all[rc])
    # Adjust layout and show the plot
    # plt.axvline(WOI +w_LL/2)
    # plt.tight_layout()
    plt.ylim([-400,600])
    plt.show()
    print(peak)

NameError: name 'plot_signals' is not defined

In [ ]:
stop

In [ ]:
sc = 6
rc = 64
lists = con_trial[(con_trial['Chan'] == rc) & (con_trial['Stim'] == sc)& (con_trial['Sig'] == 1)& (con_trial['Artefact'] <1)].reset_index(drop=True)
if len(lists)>10:
    stimnum = lists['Num'].values.astype('int')
    WOI = con_summary_all.loc[(con_summary_all.Stim == sc)&(con_summary_all.Chan ==rc), 't_WOI'].values[0]
    onset, peak, pol, _ = CCEP_onset(EEG_resp[rc, stimnum, :], WOI=0, t0=1, Fs=500, w_LL=0.25, plot=False)
    fig, ax = plot_CCEP_onset_SS(EEG_resp, con_trial, labels_all, sc, rc, onset, peak, WOI, pol, Fs = 500, t0 = 1)
    plt.ylim([-300,350])
    plt.xlabel('Time [s]')
    plt.tight_layout()
    plt.savefig(os.path.join(path, 'sleep_CCEP_'+labels_all[sc]+'_'+labels_all[rc]+'.svg'))

In [ ]:
from scipy.signal import savgol_filter, find_peaks
import LL_funcs as LLf

In [ ]:
plot_CCEP_onset_SS(EEG_resp, con_trial, labels_all, sc, rc, onset, peak_lat, WOI, polarity, Fs = 500, t0 = 1)

In [17]:
def plot_CCEP_onset_SS(EEG_resp, con_trial, labels_all, sc, rc, onset, peak_lat_general, WOI, polarity, Fs = 500, t0 = 1):
    kernel_size = int(0.1*Fs)
    kernel = np.ones(kernel_size) / kernel_size
            
    w_all = [0.25, 0.05]
    fig, axes = plt.subplots(3,1, figsize=(3, 6), sharey =True)  # Create a 2x3 subplot grid
    fig.patch.set_facecolor('xkcd:white')
    plt.suptitle(labels_all[sc]+ ' -- '+labels_all[rc])

    # Initialize variables to store the min and max y-values for each row
    row_min_max = [[np.inf, -np.inf] for _ in range(4+len(w_all))]
    
    for ix, ss in enumerate(label_sleep):
        
        lists = con_trial[(con_trial['Chan'] == rc) & (con_trial['Stim'] == sc)& (con_trial['Sig'] == 1)& (con_trial['Artefact'] <1)& (con_trial['SleepState']==ss)].reset_index(drop=True)
        stimnum = lists['Num'].values.astype('int')
        t_onsest, peak_lat,_ = CCEP_onset_finetuning(ff.lp_filter(EEG_resp[rc, stimnum, :],45,Fs), onset, peak_lat_general, polarity, WOI=WOI, t0=1, Fs=500, w_LL=0.25)
        print(peak_lat)
        data_CCEP = np.mean(ff.lp_filter(EEG_resp[rc, stimnum, :],45,Fs), 0)
        # data_CCEP = ff.lp_filter(np.mean(EEG_resp[rc, stimnum, :], 0),45,Fs)
        st = np.std(EEG_resp[rc, stimnum, :], 0)

        # Update min and max y-values for each row
        row_min_max[0] = [-300,300]
       
        # CCEP
        ax = axes[ix]
        ax.set_title(ss +', n = '+str(len(stimnum)))
        ax.plot(x_ax, data_CCEP, linewidth=2, color='k')
        ax.fill_between(x_ax, data_CCEP-st, data_CCEP+st, alpha=0.1, color='k')
        ax.axvline(0, color=[0, 0, 0])
        ax.axvline(t_onsest, color='g', label = 'onset: \n'+str(np.round(t_onsest,3))+'s')
        ax.scatter(peak_lat, data_CCEP[int((t0+peak_lat)*Fs)], color='b', label = 'peak latency: \n'+str(np.round(peak_lat,3))+'s')
        #ax.legend()
        ax.set_xlim([-0.3, 0.7])
        # ax.axvline(delay, color=[0, 1, 0])
        ax.set_ylabel('[uV]')

    return fig, axes

## Plot

In [ ]:
path

In [ ]:
sc_all = [0,  19,7, 6]
rc_all = [3,  49, 36, 26]
fig, axs = plot_CCEP_onset_pipeline(EEG_resp, con_trial, labels_all, sc_all, rc_all, CC_summ, w_LL = 0.25, Fs = 500, t0 = 1)
plt.savefig(os.path.join(path, 'subj_4cons2.svg'))

In [18]:
def plot_CCEP_onset_pipeline(EEG_resp, con_trial, labels_all, sc_all, rc_all, CC_summ, w_LL = 0.25, Fs = 500, t0 = 1):
    xlim = [-0.25, 0.5]
    ylim_CCEP = [-350, 350]
    ylim_d = [-1,1]
    fig, axes = plt.subplots(2, len(sc_all), figsize=(5.5, 2.5))  # Create a 1x3 subplot grid
    fig.patch.set_facecolor('xkcd:white')

    for ix, sc, rc in zip(np.arange(len(sc_all)), sc_all, rc_all):
        lists = con_trial[(con_trial['Chan'] == rc) & (con_trial['Stim'] == sc)& (con_trial['Sig'] == 1)& (con_trial['Artefact'] <1)].reset_index(drop=True)
        stimnum = lists['Num'].values.astype('int')
        data_CCEP = ff.lp_filter(np.mean(EEG_resp[rc, stimnum, :], 0), 45, Fs)
        CCEP_std = np.std(EEG_resp[rc, stimnum, :], 0)
        WOI = CC_summ.loc[(CC_summ.Stim == sc) & (CC_summ.Chan == rc), 't_WOI'].values[0]
        

        kernel_size = int(0.1 * Fs) + 1

        peak_lat, polarity,peak_detected = peak_latency(EEG_resp[rc, stimnum, :], WOI, t0=1, Fs=500, w_LL=0.25)
        # Calculate second derivatives using Savitzky-Golay filter
        signal_derivative = savgol_filter(data_CCEP, window_length=kernel_size, polyorder=3, deriv=2)
        pk_CCEP = data_CCEP[int((t0+peak_lat)*Fs)]
        print(peak_lat)
        #  onset
        thr = np.std(signal_derivative[int((t0-0.5)*Fs):int((t0-0.01)*Fs)])/10
        peaks, _ = find_peaks(np.abs(signal_derivative), height=thr)
        peaks = peaks[(peaks > int((t0 - 0.03) * Fs)) & (peaks < int(((t0 + peak_lat-0.01) * Fs)))]
        peak_times = peaks / Fs - t0  # Adjusting for epoched time
        t_onset =  np.max([0,peak_times[np.argmin(peak_lat-peak_times)]])# peak_times[0]
        pk_der = signal_derivative[int((t0+t_onset)*Fs)]
        # plot second derivative of LL
        ax = axes[0, ix]
        ax.plot(x_ax, data_CCEP, linewidth=2, color = 'k')
        ax.fill_between(x_ax, data_CCEP-CCEP_std,data_CCEP+CCEP_std , alpha=0.2, color = 'k')
        ax.set_title(labels_all[sc] + ' - ' + labels_all[rc])
        ax.scatter(peak_lat, pk_CCEP, color = 'b')
        ax.set_xticks([])
        if ix == 0:
            ax.set_ylabel('[uV]')
            # ax.set_yticks([0, 3, 6])
        else:
            ax.set_yticks([])
        ax.axvline(0, color=[0, 0, 0])
        ax.axvline(t_onset, color='g')
        ax.set_xlim(xlim)
        ax.set_ylim([-300,300])
        # ax.set_box_aspect(1.5 / 2)
        # Plot second derivate
        ax = axes[1, ix]
        ax.plot(x_ax, signal_derivative, linewidth=2, color = 'k')
        #ax.set_xlabel('time [s]')
        #ax.set_xticks([0, 0.5])
        if ix ==0:
            ax.set_ylabel('[uV/s^2]')
            # ax.set_yticks([0, 3, 6])
        else:
            ax.set_yticks([])
        ax.axvline(0, color=[0, 0, 0])
        ax.set_ylim(ylim_d)
        ax.scatter(t_onset, pk_der, color = 'g')
        ax.axvline(peak_lat, color='blue', linestyle='--')
        ax.set_xlim(xlim)
        # ax.set_box_aspect(1.5 / 2)


    return fig, axes

In [ ]:
def plot_CCEP_onset(EEG_resp, con_trial, labels_all, sc_all, rc_all, CC_summ, w_LL = 0.25, Fs = 500, t0 = 1):
    xlim = [-0.25, 0.5]
    ylim_CCEP = [-350, 350]
    ylim_LL = [0, 8]
    fig, axes = plt.subplots(3, len(sc_all), figsize=(len(sc_all)*2.5, 6))  # Create a 1x3 subplot grid
    fig.patch.set_facecolor('xkcd:white')

    for ix, sc, rc in zip(np.arange(len(sc_all)), sc_all, rc_all):
        lists = con_trial[(con_trial['Chan'] == rc) & (con_trial['Stim'] == sc)& (con_trial['Sig'] == 1)& (con_trial['Artefact'] <1)].reset_index(drop=True)
        stimnum = lists['Num'].values.astype('int')
        data_CCEP = ff.lp_filter(np.mean(EEG_resp[rc, stimnum, :], 0), 30, Fs)
        WOI = CC_summ.loc[(CC_summ.Stim == sc) & (CC_summ.Chan == rc), 't_WOI'].values[0]
        delay, data_LL, d1_LL, d2_LL = CCEP_func.cal_delay(data_CCEP, WOI=WOI)
        #data_CCEP = ff.lp_filter(data_CCEP, 30, Fs)
        pk_CCEP_loc = np.argmax(abs(data_CCEP[int(t0 * Fs):int((t0+WOI+0.125) * Fs)]))
        pk_CCEP = data_CCEP[int(t0*Fs+pk_CCEP_loc)]
        # plot second derivative of LL
        ax = axes[2, ix]
        ax.plot(x_ax + w_LL / 2, d2_LL, linewidth=2, alpha=0.5)
        d2_LL[d1_LL < 0] = np.nan  # only increase intresting
        d2_LL[:int((t0 - w_LL / 2) * Fs)] = np.nan  # not before Stim
        d2_LL[int((t0 * Fs) + pk_CCEP_loc):] = np.nan  # not before Stim
        ax.plot(x_ax + w_LL / 2, d2_LL, linewidth=2)
        pk_loc = np.nanargmax(d2_LL[t0 * Fs:]) / Fs
        pk = np.nanmax(d2_LL[t0 * Fs:])
        ax.plot(pk_loc + w_LL / 2, pk, 'o', color=[0, 1, 0])
        ax.set_xlabel('time [s]')
        ax.set_xticks([0, 0.5])
        if ix == 0:
            ax.set_ylabel('[uV/ms]')
            # ax.set_yticks([0, 3, 6])
        else:
            ax.set_yticks([])
        ax.axvline(0, color=[0, 0, 0])
        ax.set_xlim(xlim)
        ax.set_ylim([-0.03, 0.03])
        ax.set_box_aspect(1.5 / 2)
        # Plot LL
        ax = axes[1, ix]
        ax.plot(x_ax+w_LL/2, data_LL, linewidth=2)
        #ax.set_xlabel('time [s]')
        #ax.set_xticks([0, 0.5])
        if ix ==0:
            ax.set_ylabel('[uV/ms]')
            ax.set_yticks([0, 3, 6])
        else:
            ax.set_yticks([])
        ax.axvline(0, color=[0, 0, 0])
        ax.set_xlim(xlim)
        ax.set_ylim(ylim_LL)
        ax.axvline(pk_loc + w_LL / 2, color=[0, 1, 0])
        ax.set_box_aspect(1.5 / 2)
        # plot orignal data with a shadowd window based on peak LL
        ax = axes[0, ix]
        ax.plot(x_ax, data_CCEP, linewidth=2)
        ax.axvspan(WOI, WOI+0.25, color = [0,0,0], alpha =0.1)
        ax.set_title(labels_all[sc] + ' - ' + labels_all[rc])
        if ix ==0:
            ax.set_ylabel('[uV]')
            ax.set_yticks([-400,0,400])
        else:
            ax.set_yticks([])
        ax.axvline(0, color=[0, 0, 0])
        ax.set_xlim(xlim)
        ax.set_ylim(ylim_CCEP)
        ax.set_xticks([])
        ax.set_box_aspect(1.5 / 2)
        # ax.axvline(pk_loc, color=[0, 1,0])
        ax.axvline(delay, color=[0, 1, 0])
        ax.plot(pk_CCEP_loc/Fs, pk_CCEP, 'o', color=[1, 0, 0])


    return fig, axes

In [ ]:
CCEP_supp.plot_CCEP_onset(EEG_resp, con_trial, labels_all, sc_all, rc_all, CC_summ, w_LL = 0.25, Fs = 500, t0 = 1)

In [ ]:
stop

In [ ]:

example_surr = np.zeros((4, 2))
example_surr[0] = [3, 74]
example_surr[1] = [4, 64] #64
example_surr[2] = [65, 15] #64, [50, 75]
example_surr[3] = [49, 56] #64
example_surr = example_surr.astype('int')

example_z = np.zeros((4, 2))
example_z[0] = [53, 0]
example_z[1] = [53, 76] #64
example_z[2] = [57, 20] #64[54, 68]
example_z[3] = [55, 49] #64
example_z = example_z.astype('int')
# Call the function with your data
fig, axes = CCEP_plot.plot_SigCon_examples(example_z[:,0], example_z[:,1], con_trial, EEG_resp,labels_all, 0)
# plt.savefig('X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\CCEP\CCEP_sigCon_example_zscore.svg')